# Macroeconomic Stress Testing for Retail Loan Defaults
**Course Alignment:** ES 335 Machine Learning (IIT Gandhinagar)  
**Target Career Domain:** Quantitative Risk & Analytics (Barclays, J.P. Morgan, Goldman Sachs)  
**Core Research Question:** Do macroeconomic indicators (unemployment rate, inflation CPI, Fed funds rate) significantly improve retail loan default prediction compared to traditional models relying strictly on individual borrower credit profiles (FICO, DTI, Income)?

---
### End-to-End Quantitative Pipeline Overview
1. **Phase 1:** Environment Setup & Toolchain Initialization
2. **Phase 2:** Data Procurement (LendingClub Micro-data & FRED Macro Series)
3. **Phase 3:** Data Cleaning & Point-in-Time (PIT) Chronological Merging (`pd.merge_asof`)
4. **Phase 4:** Feature Engineering & Preprocessing (Macro-Borrower Interaction Terms)
5. **Phase 5:** Model Training & Out-of-Time (OOT) Validation (Chronological Splits)
6. **Phase 6:** Evaluation Metrics & Probability Calibration (PR-AUC, ROC-AUC, Brier Score)
7. **Phase 7:** Model Coefficient Export for Static GitHub Pages Interactive Application

## Phase 1: Environment Setup & Toolchain Initialization

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling & Econometrics
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
print('All primary libraries imported successfully.')

## Phase 2: Data Procurement (Raw Materials)
- **Borrower Micro-Data:** LendingClub Loan Dataset (origination dates `issue_d`, FICO, DTI, Income, target `loan_status`)
- **Macroeconomic Series:** FRED API (`UNRATE`, `CPIAUCSL`, `FEDFUNDS`)

In [ ]:
# Ingest Macroeconomic Series via FRED API or Local Cache
from fredapi import Fred
from dotenv import load_dotenv

load_dotenv()
fred_api_key = os.getenv('FRED_API_KEY')

if fred_api_key:
    fred = Fred(api_key=fred_api_key)
    print('FRED API connection established.')
    # Example series retrieval:
    # unrate = fred.get_series('UNRATE')
    # cpi = fred.get_series('CPIAUCSL')
    # fedfunds = fred.get_series('FEDFUNDS')
else:
    print('Note: FRED_API_KEY not configured in .env. You can also load cached macro CSVs.')

## Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging
Key integrity requirements:
1. **Target Filtering:** Retain only completed outcomes: `Charged Off` (1) and `Fully Paid` (0). Exclude ongoing `Current` loans.
2. **Eliminate Future Leakage:** Drop post-origination columns (`recoveries`, `total_pymnt`, `last_pymnt_d`, etc.).
3. **Publication Lag:** Shift macroeconomic indicators by 1 month (`macro_df['UNRATE'].shift(1)`).
4. **PIT Merge:** Perform exact 'as-of' backward merge using `pd.merge_asof` on loan `issue_d`.

In [ ]:
# TODO: Load LendingClub data and perform PIT merge with macro series

## Phase 4: Feature Engineering & Preprocessing
- Construct Macro-Borrower Interaction Terms:
  - `dti_x_inflation = DTI * CPI_rate`
  - `income_to_unemp = Annual_Income / Unemployment_Rate`
- Median Imputation for missing values
- Standard scaling fitted strictly on training data

In [ ]:
# TODO: Feature engineering and strictly partitioned scaling

## Phase 5: Model Training & Out-of-Time (OOT) Validation
- Chronological Split: Train on 2013–2016 loans, Test/Validate on 2017 loans (Zero future look-ahead bias).
- Econometric Baseline: `statsmodels.api.Logit` to determine p-values and logit odds ratios.
- Machine Learning Model: `scikit-learn` `LogisticRegression(class_weight='balanced')`.

In [ ]:
# TODO: Chronological split and model fitting

## Phase 6: Evaluation Metrics & Probability Calibration
- PR-AUC (Precision-Recall Area Under Curve) as primary metric for class imbalance.
- ROC-AUC and Brier Score Loss $BS = \frac{1}{N} \sum (f_i - y_i)^2$.
- Probability Calibration Curves (Reliability diagrams).

In [ ]:
# TODO: Calculate evaluation metrics and plot calibration curves

## Phase 7: ES 335 Deliverable — Static Web Model Export
- Extract model coefficients (`model.coef_`) and intercept (`model.intercept_`).
- Export to `model_coefficients.json` for client-side JavaScript probability calculation:  
  $$P = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i X_i)}}$$

In [ ]:
# TODO: Export weights to model_coefficients.json for GitHub Pages web app